# Conditional normalizing flows for ATLAS jet simulation

This notebook trains a conditional autoregressive rational-quadratic spline flow to
generate reconstructed ATLAS jet features from truth-level jet information. The generated
events are validated with feature distributions, correlation matrices, and reconstructed
$W$-boson and top-quark masses.

The workflow uses the public
[ATLAS $t\bar{t}$ JetSet dataset](https://opendata.cern.ch/record/atlas-93940).
If the parquet input is unavailable, the notebook automatically creates a small correlated
toy dataset so that the complete code path can be smoke-tested.


## 1. Environment and reproducibility

The full training configuration is used when `DATA_PATH` exists. When toy data are used,
the model and training duration are reduced automatically for a quick functional test.


In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from itertools import combinations
from pathlib import Path
import random

import matplotlib.pyplot as plt
import normflows as nf
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from scipy.stats import ks_2samp
from torch.utils.data import DataLoader, TensorDataset
from tqdm.auto import trange

%config InlineBackend.figure_format = "svg"

SEED = 121
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
    torch.cuda.manual_seed_all(SEED)
elif torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")

# Warn instead of failing if a backend lacks a deterministic implementation.
torch.use_deterministic_algorithms(True, warn_only=True)
if torch.cuda.is_available():
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

print(f"Using device: {DEVICE}")


## 2. Configuration

`jets.parquet` is intentionally excluded from version control. Set `DATA_PATH` to the
prepared JetSet parquet file. The expected source columns are validated before
preprocessing begins.


In [ ]:
DATA_PATH = Path("../data/jets.parquet")
FIGURE_DIR = Path("../figures")
SAVE_FIGURES = True

JETS_PER_EVENT = 6
TRAIN_FRACTION = 0.8
TOY_EVENT_COUNT = 256

RECO_SOURCE_COLUMNS = [
    "eventNumber",
    "pt",
    "eta",
    "phi",
    "mass",
    "GN2v01_pb",
    "GN2v01_pc",
    "GN2v01_pu",
    "GN2v01_ptau",
]
TRUTH_SOURCE_COLUMNS = [
    "ptFromTruthJet",
    "etaFromTruthJet",
    "phiFromTruthJet",
    "mFromTruthJet",
]

TARGET_FEATURES = ["pt", "eta", "phi", "mass", "zb", "zc", "ztau"]
CONTEXT_FEATURES = ["pt_truth", "eta_truth", "phi_truth", "m_truth"]
PHYSICAL_FEATURES = ["pt", "eta", "phi", "mass", "pb", "pc", "pu", "ptau"]

USE_TOY_DATA = not DATA_PATH.exists()
if USE_TOY_DATA:
    print(f"{DATA_PATH} was not found; using a small toy dataset for a smoke test.")


## 3. Data loading and preprocessing

The toy generator below preserves the input schema and introduces correlations between
truth and reconstructed kinematics. It is a software test only and is not a physics model.

For either data source, preprocessing:

- converts momenta and masses from MeV to GeV;
- applies $p_T > 30$ GeV and $|\eta| < 2.5$;
- retains events with exactly six selected jets;
- converts four flavour probabilities to three log-ratios relative to $p_u$;
- sorts jets within each event by decreasing reconstructed $p_T$; and
- splits at event level to prevent event leakage.


In [ ]:
def make_toy_jet_data(n_events: int = TOY_EVENT_COUNT, seed: int = SEED) -> pd.DataFrame:
    """Create correlated six-jet events with the same columns as the analysis input."""
    rng = np.random.default_rng(seed)
    n_rows = n_events * JETS_PER_EVENT

    event_number = np.repeat(np.arange(1_000_000, 1_000_000 + n_events), JETS_PER_EVENT)
    truth_pt_gev = rng.lognormal(mean=np.log(85.0), sigma=0.35, size=n_rows) + 35.0
    truth_eta = rng.uniform(-2.2, 2.2, size=n_rows)
    truth_phi = rng.uniform(-np.pi, np.pi, size=n_rows)
    truth_mass_gev = rng.lognormal(mean=np.log(12.0), sigma=0.35, size=n_rows)

    reco_pt_gev = np.clip(truth_pt_gev * rng.normal(1.0, 0.08, size=n_rows), 31.0, None)
    reco_eta = np.clip(truth_eta + rng.normal(0.0, 0.05, size=n_rows), -2.45, 2.45)
    reco_phi = np.angle(np.exp(1j * (truth_phi + rng.normal(0.0, 0.05, size=n_rows))))
    reco_mass_gev = np.clip(
        truth_mass_gev * rng.normal(1.05, 0.15, size=n_rows), 0.2, None
    )

    # Dirichlet draws create valid, non-degenerate flavour probabilities.
    flavour_class = rng.integers(0, 4, size=n_rows)
    concentration = np.full((n_rows, 4), 0.8)
    concentration[np.arange(n_rows), flavour_class] = 8.0
    probabilities = np.vstack([rng.dirichlet(alpha) for alpha in concentration])

    return pd.DataFrame(
        {
            "eventNumber": event_number,
            "pt": reco_pt_gev * 1_000.0,
            "eta": reco_eta,
            "phi": reco_phi,
            "mass": reco_mass_gev * 1_000.0,
            "GN2v01_pb": probabilities[:, 0],
            "GN2v01_pc": probabilities[:, 1],
            "GN2v01_pu": probabilities[:, 2],
            "GN2v01_ptau": probabilities[:, 3],
            "ptFromTruthJet": truth_pt_gev * 1_000.0,
            "etaFromTruthJet": truth_eta,
            "phiFromTruthJet": truth_phi,
            "mFromTruthJet": truth_mass_gev * 1_000.0,
        }
    )


def load_jet_data(path: Path, use_toy_data: bool = False) -> pd.DataFrame:
    """Load the prepared parquet dataset or return a schema-compatible toy sample."""
    if use_toy_data:
        return make_toy_jet_data()
    return pd.read_parquet(path)


@dataclass
class PreparedEvents:
    """Event-level arrays and jet tables used for training and evaluation."""

    x_train: np.ndarray
    x_test: np.ndarray
    c_train: np.ndarray
    c_test: np.ndarray
    train_jets: pd.DataFrame
    test_jets: pd.DataFrame


def _event_matrix(frame: pd.DataFrame, columns: list[str]) -> np.ndarray:
    """Convert sorted jet rows into one fixed-length vector per event."""
    values = frame[columns].to_numpy(dtype=np.float32)
    return values.reshape(-1, JETS_PER_EVENT * len(columns))


def prepare_events(raw: pd.DataFrame, seed: int = SEED) -> PreparedEvents:
    """Validate, select, transform, sort, and split jet rows at event level."""
    required = set(RECO_SOURCE_COLUMNS + TRUTH_SOURCE_COLUMNS)
    missing = sorted(required.difference(raw.columns))
    if missing:
        raise ValueError(f"Input data are missing required columns: {missing}")

    jets = raw[RECO_SOURCE_COLUMNS + TRUTH_SOURCE_COLUMNS].dropna().copy()
    for column in ["pt", "mass", "ptFromTruthJet", "mFromTruthJet"]:
        jets[column] = jets[column] / 1_000.0

    jets = jets[(jets["pt"] > 30.0) & (jets["eta"].abs() < 2.5)].copy()
    event_sizes = jets.groupby("eventNumber").size()
    selected_events = event_sizes[event_sizes == JETS_PER_EVENT].index
    jets = jets[jets["eventNumber"].isin(selected_events)].copy()
    if jets.empty:
        raise ValueError("No events contain exactly six jets after the kinematic selection.")

    jets = jets.rename(
        columns={
            "GN2v01_pb": "pb",
            "GN2v01_pc": "pc",
            "GN2v01_pu": "pu",
            "GN2v01_ptau": "ptau",
            "ptFromTruthJet": "pt_truth",
            "etaFromTruthJet": "eta_truth",
            "phiFromTruthJet": "phi_truth",
            "mFromTruthJet": "m_truth",
        }
    )

    probability_columns = ["pb", "pc", "pu", "ptau"]
    probabilities = jets[probability_columns].astype(np.float64)
    probabilities = probabilities.div(probabilities.sum(axis=1), axis=0)
    probabilities = probabilities.clip(1e-6, 1.0 - 1e-6)
    probabilities = probabilities.div(probabilities.sum(axis=1), axis=0)
    jets[probability_columns] = probabilities

    jets["zb"] = np.log(jets["pb"] / jets["pu"])
    jets["zc"] = np.log(jets["pc"] / jets["pu"])
    jets["ztau"] = np.log(jets["ptau"] / jets["pu"])
    jets = jets.sort_values(
        ["eventNumber", "pt"], ascending=[True, False]
    ).reset_index(drop=True)

    event_ids = jets["eventNumber"].unique()
    rng = np.random.default_rng(seed)
    rng.shuffle(event_ids)
    split_index = int(TRAIN_FRACTION * len(event_ids))
    if split_index == 0 or split_index == len(event_ids):
        raise ValueError("At least two selected events are required for a train/test split.")

    train_ids = set(event_ids[:split_index])
    test_ids = set(event_ids[split_index:])
    train_jets = jets[jets["eventNumber"].isin(train_ids)].copy()
    test_jets = jets[jets["eventNumber"].isin(test_ids)].copy()

    return PreparedEvents(
        x_train=_event_matrix(train_jets, TARGET_FEATURES),
        x_test=_event_matrix(test_jets, TARGET_FEATURES),
        c_train=_event_matrix(train_jets, CONTEXT_FEATURES),
        c_test=_event_matrix(test_jets, CONTEXT_FEATURES),
        train_jets=train_jets,
        test_jets=test_jets,
    )


In [ ]:
raw_jets = load_jet_data(DATA_PATH, use_toy_data=USE_TOY_DATA)
data = prepare_events(raw_jets)

print(f"Training events: {len(data.x_train):,}")
print(f"Testing events:  {len(data.x_test):,}")
print(f"Target shape:    {data.x_train.shape}")
print(f"Context shape:   {data.c_train.shape}")


## 4. Physics reconstruction utilities

The two jets with the largest flavour discriminant are treated as the $b$-jet candidates.
The remaining four jets are paired into two $W$ candidates by minimizing their combined
mass consistency with $m_W=80.4$ GeV. The two possible assignments of the $b$ jets to the
$W$ candidates are then compared using the agreement of the two reconstructed top masses.


In [ ]:
def add_four_vectors(frame: pd.DataFrame) -> pd.DataFrame:
    """Return a copy with Cartesian four-vector components in GeV."""
    result = frame.copy()
    result["px"] = result["pt"] * np.cos(result["phi"])
    result["py"] = result["pt"] * np.sin(result["phi"])
    result["pz"] = result["pt"] * np.sinh(result["eta"])
    result["E"] = np.sqrt(
        result["px"] ** 2
        + result["py"] ** 2
        + result["pz"] ** 2
        + result["mass"] ** 2
    )
    return result


def invariant_mass(vectors: np.ndarray) -> float:
    """Compute the invariant mass of rows ordered as E, px, py, pz."""
    energy, px, py, pz = vectors.sum(axis=0)
    mass_squared = energy**2 - px**2 - py**2 - pz**2
    return float(np.sqrt(max(mass_squared, 0.0)))


def best_w_pairs(light_jets: np.ndarray, w_mass: float = 80.4) -> tuple:
    """Find two disjoint jet pairs closest to the expected W-boson mass."""
    if len(light_jets) < 4:
        raise ValueError("At least four light-jet candidates are required.")

    best_score = np.inf
    best_pairing = None
    for indices in combinations(range(len(light_jets)), 4):
        a, b, c, d = indices
        pairings = [
            ((a, b), (c, d)),
            ((a, c), (b, d)),
            ((a, d), (b, c)),
        ]
        for first, second in pairings:
            m_first = invariant_mass(light_jets[list(first), 1:5])
            m_second = invariant_mass(light_jets[list(second), 1:5])
            score = ((m_first - w_mass) / 10.0) ** 2 + (
                (m_second - w_mass) / 10.0
            ) ** 2
            if score < best_score:
                best_score = score
                best_pairing = (first, second)

    first, second = best_pairing
    return light_jets[list(first)], light_jets[list(second)]


def reconstruct_top_masses(event_jets: pd.DataFrame) -> tuple[float, float, float, float]:
    """Reconstruct two top and two W masses from one six-jet event."""
    if len(event_jets) < JETS_PER_EVENT:
        return (np.nan, np.nan, np.nan, np.nan)

    columns = ["pt", "E", "px", "py", "pz", "pb", "pc", "pu", "ptau"]
    jets = event_jets[columns].to_numpy(dtype=float)
    discriminator = jets[:, 5] / np.clip(jets[:, 6:9].sum(axis=1), 1e-12, None)
    b_indices = np.argsort(discriminator)[-2:]
    b_jets = jets[b_indices]

    light_jets = np.delete(jets, b_indices, axis=0)
    light_jets = light_jets[np.argsort(light_jets[:, 0])[::-1]]
    w_first, w_second = best_w_pairs(light_jets)

    w_mass_first = invariant_mass(w_first[:, 1:5])
    w_mass_second = invariant_mass(w_second[:, 1:5])

    def b_plus_w_mass(b_jet: np.ndarray, w_jets: np.ndarray) -> float:
        return invariant_mass(
            np.vstack([b_jet[1:5], w_jets[:, 1:5]])
        )

    # Assignment A: b1+W1 and b2+W2. Assignment B swaps the W candidates.
    assignment_a = (
        b_plus_w_mass(b_jets[0], w_first),
        b_plus_w_mass(b_jets[1], w_second),
    )
    assignment_b = (
        b_plus_w_mass(b_jets[0], w_second),
        b_plus_w_mass(b_jets[1], w_first),
    )
    score_a = ((assignment_a[0] - assignment_a[1]) / 25.0) ** 2
    score_b = ((assignment_b[0] - assignment_b[1]) / 25.0) ** 2
    top_first, top_second = assignment_a if score_a <= score_b else assignment_b

    return top_first, top_second, w_mass_first, w_mass_second


def reconstruct_event_sample(frame: pd.DataFrame) -> tuple[np.ndarray, np.ndarray]:
    """Reconstruct top and W masses for every event in a jet table."""
    top_masses, w_masses = [], []
    with_vectors = add_four_vectors(frame)
    for _, event in with_vectors.groupby("eventNumber", sort=False):
        top_first, top_second, w_first, w_second = reconstruct_top_masses(event)
        top_masses.extend([top_first, top_second])
        w_masses.extend([w_first, w_second])
    return np.asarray(top_masses), np.asarray(w_masses)


## 5. Conditional spline-flow model

The production configuration uses 12 autoregressive rational-quadratic spline blocks with
permutation layers between them. Reconstructed targets are standardized using training-set
statistics. Truth-level event vectors are supplied as the condition to every spline block.


In [ ]:
class ConditionalSplineFlow:
    """Conditional autoregressive rational-quadratic spline normalizing flow."""

    def __init__(
        self,
        reco_events: np.ndarray,
        truth_events: np.ndarray,
        *,
        learning_rate: float = 1e-6,
        weight_decay: float = 5e-5,
        hidden_units: int = 128,
        hidden_layers: int = 3,
        flow_blocks: int = 12,
        num_bins: int = 16,
        tail_bound: float = 3.0,
        batch_size: int = 1024,
        device: torch.device = DEVICE,
        seed: int = SEED,
    ) -> None:
        self.device = device
        self.dimension = reco_events.shape[1]
        self.context_dimension = truth_events.shape[1]

        targets = torch.as_tensor(reco_events, dtype=torch.float32)
        contexts = torch.as_tensor(truth_events, dtype=torch.float32)
        self.target_mean = targets.mean(dim=0, keepdim=True)
        self.target_std = targets.std(dim=0, keepdim=True)
        self.target_std = torch.where(
            self.target_std < 1e-6,
            torch.ones_like(self.target_std),
            self.target_std,
        )
        standardized_targets = (targets - self.target_mean) / self.target_std

        generator = torch.Generator().manual_seed(seed)
        self.loader = DataLoader(
            TensorDataset(standardized_targets, contexts),
            batch_size=batch_size,
            shuffle=True,
            num_workers=0,
            generator=generator,
        )

        base = nf.distributions.base.DiagGaussian(self.dimension)
        transforms = []
        for _ in range(flow_blocks):
            transforms.append(
                nf.flows.AutoregressiveRationalQuadraticSpline(
                    self.dimension,
                    hidden_layers,
                    hidden_units,
                    num_bins=num_bins,
                    tail_bound=tail_bound,
                    num_context_channels=self.context_dimension,
                )
            )
            transforms.append(nf.flows.LULinearPermute(self.dimension))

        self.model = nf.ConditionalNormalizingFlow(base, transforms).to(self.device)
        self.optimizer = torch.optim.Adam(
            self.model.parameters(),
            lr=learning_rate,
            weight_decay=weight_decay,
        )
        self.loss_history: list[float] = []

    def fit(self, epochs: int = 100, show_progress: bool = True) -> list[float]:
        """Minimize the forward KL divergence and return mean epoch losses."""
        self.model.train()
        iterator = trange(epochs, desc="Training", disable=not show_progress)
        for _ in iterator:
            total_loss = 0.0
            batch_count = 0
            for batch_targets, batch_contexts in self.loader:
                batch_targets = batch_targets.to(self.device)
                batch_contexts = batch_contexts.to(self.device)

                self.optimizer.zero_grad(set_to_none=True)
                loss = self.model.forward_kld(
                    batch_targets, context=batch_contexts
                )
                loss.backward()
                self.optimizer.step()

                total_loss += float(loss.detach().cpu())
                batch_count += 1

            mean_loss = total_loss / max(batch_count, 1)
            self.loss_history.append(mean_loss)
            iterator.set_postfix(loss=f"{mean_loss:.3f}")
        return self.loss_history

    def sample(self, truth_events: np.ndarray, batch_size: int = 256) -> np.ndarray:
        """Generate reconstructed event vectors conditioned on truth-level events."""
        self.model.eval()
        contexts = torch.as_tensor(
            truth_events, dtype=torch.float32, device=self.device
        )
        samples = []
        with torch.inference_mode():
            for start in range(0, len(contexts), batch_size):
                context_batch = contexts[start : start + batch_size]
                batch, _ = self.model.sample(
                    num_samples=len(context_batch), context=context_batch
                )
                samples.append(batch.cpu())

        standardized = torch.cat(samples, dim=0)
        physical_scale = standardized * self.target_std + self.target_mean
        return physical_scale.numpy()


In [ ]:
# Use the thesis configuration for real data and a compact model for the toy smoke test.
model_config = {
    "learning_rate": 1e-6,
    "weight_decay": 5e-5,
    "hidden_units": 32 if USE_TOY_DATA else 128,
    "hidden_layers": 2 if USE_TOY_DATA else 3,
    "flow_blocks": 2 if USE_TOY_DATA else 12,
    "num_bins": 8 if USE_TOY_DATA else 16,
    "tail_bound": 3.0,
    "batch_size": 128 if USE_TOY_DATA else 1024,
}
training_epochs = 2 if USE_TOY_DATA else 100

flow = ConditionalSplineFlow(data.x_train, data.c_train, **model_config)
loss_history = flow.fit(epochs=training_epochs)

plt.figure(figsize=(6, 4))
plt.plot(loss_history)
plt.xlabel("Epoch")
plt.ylabel("Mean forward-KL loss")
plt.title("Training loss")
plt.tight_layout()
plt.show()


## 6. Generate events and restore physical features

The flow predicts $(p_T,\eta,\phi,m,z_b,z_c,z_\tau)$ for each jet. The three log-ratios
are mapped back to the four normalized flavour probabilities before physics validation.


In [ ]:
def restore_probabilities(frame: pd.DataFrame) -> pd.DataFrame:
    """Convert flavour log-ratios back to four probabilities that sum to one."""
    result = frame.copy()
    ratios = np.exp(
        np.clip(result[["zb", "zc", "ztau"]].to_numpy(), -30.0, 30.0)
    )
    denominator = 1.0 + ratios.sum(axis=1)
    result["pu"] = 1.0 / denominator
    result["pb"] = ratios[:, 0] / denominator
    result["pc"] = ratios[:, 1] / denominator
    result["ptau"] = ratios[:, 2] / denominator
    return result.drop(columns=["zb", "zc", "ztau"])


generated_events = flow.sample(data.c_test)
generated_jets = pd.DataFrame(
    generated_events.reshape(-1, len(TARGET_FEATURES)),
    columns=TARGET_FEATURES,
)
generated_jets["eventNumber"] = data.test_jets["eventNumber"].to_numpy()
generated_jets = restore_probabilities(generated_jets)

test_jets = data.test_jets[
    ["eventNumber"] + TARGET_FEATURES + ["pb", "pc", "pu", "ptau"]
].copy()
test_jets = test_jets.drop(columns=["zb", "zc", "ztau"])

generated_top_masses, generated_w_masses = reconstruct_event_sample(generated_jets)
test_top_masses, test_w_masses = reconstruct_event_sample(test_jets)

top_ks = ks_2samp(test_top_masses, generated_top_masses).statistic
w_ks = ks_2samp(test_w_masses, generated_w_masses).statistic
print(f"Top-mass KS statistic: {top_ks:.4f}")
print(f"W-mass KS statistic:   {w_ks:.4f}")


## 7. Diagnostic plots

These plots test marginal distributions, pairwise correlations, and downstream
reconstructed masses. Toy-data results only confirm that the software runs; physics
conclusions must use the full JetSet sample and production model configuration.


In [ ]:
def save_figure(figure: plt.Figure, filename: str) -> None:
    """Save a publication-quality SVG when figure output is enabled."""
    if SAVE_FIGURES:
        FIGURE_DIR.mkdir(parents=True, exist_ok=True)
        figure.savefig(FIGURE_DIR / filename, bbox_inches="tight")


def plot_feature_distributions(
    generated: pd.DataFrame, reference: pd.DataFrame
) -> plt.Figure:
    """Compare one-dimensional reconstructed feature distributions."""
    limits = {
        "pt": (0, 500),
        "eta": (-2.6, 2.6),
        "phi": (-np.pi, np.pi),
        "mass": (0, 80),
        "pb": (0, 1),
        "pc": (0, 1),
        "pu": (0, 1),
        "ptau": (0, 1),
    }
    figure, axes = plt.subplots(2, 4, figsize=(16, 7))
    for axis, variable in zip(axes.flat, PHYSICAL_FEATURES):
        ref_values = reference[variable].to_numpy()
        gen_values = generated[variable].to_numpy()
        ks_value = ks_2samp(ref_values, gen_values).statistic
        bins = np.linspace(*limits[variable], 80)
        axis.hist(ref_values, bins=bins, density=True, histtype="step", label="Test")
        axis.hist(
            gen_values,
            bins=bins,
            density=True,
            histtype="step",
            linestyle="--",
            label="Generated",
        )
        axis.set_title(f"{variable}: KS={ks_value:.3f}")
        axis.set_xlim(limits[variable])
        axis.set_ylabel("Density")
    axes[0, 0].legend()
    figure.tight_layout()
    return figure


def plot_correlations(
    generated: pd.DataFrame, reference: pd.DataFrame
) -> plt.Figure:
    """Plot reference, generated, and residual Pearson correlations."""
    reference_corr = reference[PHYSICAL_FEATURES].corr()
    generated_corr = generated[PHYSICAL_FEATURES].corr()
    difference = generated_corr - reference_corr

    figure, axes = plt.subplots(1, 3, figsize=(17, 5))
    sns.heatmap(reference_corr, vmin=-1, vmax=1, cmap="coolwarm", ax=axes[0])
    sns.heatmap(generated_corr, vmin=-1, vmax=1, cmap="coolwarm", ax=axes[1])
    residual_limit = max(float(np.abs(difference.to_numpy()).max()), 0.05)
    sns.heatmap(
        difference,
        vmin=-residual_limit,
        vmax=residual_limit,
        cmap="coolwarm",
        ax=axes[2],
    )
    axes[0].set_title("Test jets")
    axes[1].set_title("Generated jets")
    axes[2].set_title("Generated - test")
    figure.tight_layout()
    return figure


def plot_mass_distributions() -> plt.Figure:
    """Compare reconstructed top- and W-mass distributions."""
    figure, axes = plt.subplots(1, 2, figsize=(12, 4.5))
    axes[0].hist(
        test_top_masses,
        bins=100,
        range=(0, 600),
        density=True,
        histtype="step",
        label="Test",
    )
    axes[0].hist(
        generated_top_masses,
        bins=100,
        range=(0, 600),
        density=True,
        histtype="step",
        linestyle="--",
        label="Generated",
    )
    axes[0].axvline(172.5, color="black", linestyle=":", label="172.5 GeV")
    axes[0].set_title(f"Top mass: KS={top_ks:.3f}")
    axes[0].set_xlabel("Mass [GeV]")
    axes[0].set_ylabel("Density")
    axes[0].legend()

    axes[1].hist(
        test_w_masses,
        bins=80,
        range=(0, 250),
        density=True,
        histtype="step",
        label="Test",
    )
    axes[1].hist(
        generated_w_masses,
        bins=80,
        range=(0, 250),
        density=True,
        histtype="step",
        linestyle="--",
        label="Generated",
    )
    axes[1].axvline(80.4, color="black", linestyle=":", label="80.4 GeV")
    axes[1].set_title(f"W mass: KS={w_ks:.3f}")
    axes[1].set_xlabel("Mass [GeV]")
    axes[1].set_ylabel("Density")
    axes[1].legend()
    figure.tight_layout()
    return figure


feature_figure = plot_feature_distributions(generated_jets, test_jets)
save_figure(feature_figure, "feature-distributions.svg")
plt.show()

correlation_figure = plot_correlations(generated_jets, test_jets)
save_figure(correlation_figure, "correlation-comparison.svg")
plt.show()

mass_figure = plot_mass_distributions()
save_figure(mass_figure, "mass-comparison.svg")
plt.show()


## 8. Interpretation

For the full dataset, compare marginal KS statistics, the residual correlation matrix, and
reconstructed mass distributions together. Good one-dimensional agreement alone does not
demonstrate that a generative model preserves event-level physics. The reconstructed masses
provide a downstream test of the correlations learned across jets.
